# MELD Phase3 Pipeline with Llama Final Aggregator

This notebook mirrors the phase3 pipeline, but replaces only the final council aggregator with the Llama Vertex endpoint used in llama3_plain_validation.

In [4]:
import os
import sys
import json
import re
import concurrent.futures

import pandas as pd
import vertexai
from dotenv import load_dotenv
from vertexai.generative_models import GenerativeModel

sys.path.append('../')
from src.load_data import *
from src.agents import *
from src.fine_tuned_agents_phase3 import *

load_dotenv()

True

In [5]:
testing_data = load_data_from_csv('../data/test_sent_emo.csv')
print(testing_data.head())
testing_data.drop_duplicates(inplace=True)

Data loaded successfully from ../data/test_sent_emo.csv
   Sr No.                                          Utterance Speaker  \
0       1  Why do all youre coffee mugs have numbers on ...    Mark   
1       2  Oh. Thats so Monica can keep track. That way ...  Rachel   
2       3                                       Y'know what?  Rachel   
3      19                     Come on, Lydia, you can do it.    Joey   
4      20                                              Push!    Joey   

    Emotion Sentiment  Dialogue_ID  Utterance_ID  Season  Episode  \
0  surprise  positive            0             0       3       19   
1     anger  negative            0             1       3       19   
2   neutral   neutral            0             2       3       19   
3   neutral   neutral            1             0       1       23   
4       joy  positive            1             1       1       23   

      StartTime       EndTime  
0  00:14:38,127  00:14:40,378  
1  00:14:40,629  00:14:47,385  


In [6]:
# Llama endpoint setup for council aggregation
PROJECT_ID = os.getenv('LLAMA_MODEL_PROJECT_ID') or os.getenv('TUNED_MODEL_PROJECT_ID')
LOCATION = os.getenv('VERTEX_LOCATION', 'us-central1')
ENDPOINT_ID = '2346569469662330880'

if not PROJECT_ID:
    raise ValueError('Missing project id. Set LLAMA_MODEL_PROJECT_ID or TUNED_MODEL_PROJECT_ID in .env')

vertexai.init(project=PROJECT_ID, location=LOCATION)
llama3_model = GenerativeModel(f'projects/{PROJECT_ID}/locations/{LOCATION}/endpoints/{ENDPOINT_ID}')

def call_llama_council_aggregator(recognition_id, utterance, context, profile, sentiment, dynamics, emotional_shift, speaker_bio_card=None, previous_predictions=None):
    """Phase3 council prompt/inputs, with Llama used as the final aggregator model."""
    bio_card_content = speaker_bio_card if speaker_bio_card else '[No speaker persona available]'

    previous_context_content = ''
    if previous_predictions and len(previous_predictions) > 0:
        previous_context_content = 'Last 3 Predictions:\n'
        for i, pred in enumerate(previous_predictions, 1):
            prev_utterance = pred.get('utterance', '[N/A]')[:80]
            prev_emotion = pred.get('emotion', 'unknown')
            prev_confidence = pred.get('confidence', 'N/A')
            prev_shift = pred.get('shift', 'FALSE')

            confidence_flag = ''
            try:
                conf_value = float(prev_confidence) if isinstance(prev_confidence, (int, float)) else 0.0
                if conf_value < 0.60:
                    confidence_flag = ' ⚠️ LOW CONFIDENCE'
            except (ValueError, TypeError):
                pass

            shift_info = '[SHIFT: TRUE - Pivot Detected]' if prev_shift == 'TRUE' else '[SHIFT: FALSE - Continuity]'
            previous_context_content += f"{i}. '{prev_utterance}...'\n   Emotion: {prev_emotion} | Confidence: {prev_confidence}{confidence_flag} | {shift_info}\n"
    else:
        previous_context_content = '[No previous predictions available - First utterance]'

    prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Role: Chief Justice & Emotion Arbiter (Fine-Tuned Expert)

Task: Provide the final emotion classification for the target utterance. 
You are an expert fine-tuned on the MELD dataset. Use the provided Bio Card and Agent Reports to inform your decision, but ultimately rely on your specialized training to resolve conflicts.

Guidelines:
1. Label Space: Choose exactly one: [anger, disgust, fear, joy, neutral, sadness, surprise].
2. Persona Integration: Use the Bio Card to calibrate arousal (e.g., Monica's intensity vs. Chandler's sarcasm).
3. Conflict Resolution: If Agent Reports conflict with your internal fine-tuned logic, prioritize your own expertise unless the reports provide clear lexical or temporal evidence you missed.
4. Output: You must return a valid JSON object containing the recognition_id, a brief one-sentence reasoning, and the predicted_emotion.<|eot_id|>

<|start_header_id|>user<|end_header_id|>
[SPEAKER BIO CARD WILL BE PROVIDED HERE]

[RECENT COUNCIL DECISIONS WILL BE PROVIDED HERE] """
    prompt = prompt.replace('[SPEAKER BIO CARD WILL BE PROVIDED HERE]', bio_card_content)
    prompt = prompt.replace('[RECENT COUNCIL DECISIONS WILL BE PROVIDED HERE]', previous_context_content)
    prompt += f"""

════════════════════════════════════════════════════════════
CURRENT UTTERANCE ANALYSIS
════════════════════════════════════════════════════════════

Recognition_ID: {recognition_id}
Target Utterance: \"{utterance}\"
Current Emotional Shift Status: {emotional_shift}

EXPERT REPORTS:
1. Context Historian: {context}
2. Character Profiler: {profile}
3. Sentiment Analyst: {sentiment}
4. Social Dynamics: {dynamics}
5. Emotional Shift Detector: {emotional_shift}
"""

    response = llama3_model.generate_content(prompt)
    return response.text

In [7]:
# Load speaker bio cards for council context
def load_speaker_bio_cards(json_file='../logs/speaker_bio_cards.json'):
    """Load bio cards indexed by speaker name."""
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            bio_cards_data = json.load(f)
        print(f"Loaded {len(bio_cards_data)} speaker bio cards")
        return bio_cards_data
    except FileNotFoundError:
        print(f"Bio cards file not found at {json_file}. Proceeding without speaker context.")
        return {}
    except json.JSONDecodeError:
        print('Error decoding bio cards JSON. Proceeding without speaker context.')
        return {}

def get_speaker_bio_card_text(speaker_name, bio_cards_dict):
    """Get formatted bio card text for a speaker."""
    if not bio_cards_dict or speaker_name not in bio_cards_dict:
        return None

    bio_card = bio_cards_dict[speaker_name]

    if isinstance(bio_card, dict):
        try:
            lines = [f"Speaker: {bio_card.get('speaker', speaker_name)}"]
            if 'static_persona' in bio_card:
                lines.append(f"Persona: {bio_card['static_persona']}")
            if 'linguistic_style' in bio_card:
                lines.append(f"Linguistic Style: {bio_card['linguistic_style']}")
            if 'baseline_arousal' in bio_card:
                lines.append(f"Baseline Arousal: {bio_card['baseline_arousal']}")
            if 'negative_expression' in bio_card:
                lines.append(f"Negative Expression: {bio_card['negative_expression']}")
            if 'social_relationship_priors' in bio_card:
                lines.append(f"Social Relations: {bio_card['social_relationship_priors']}")
            return '\n'.join(lines)
        except Exception:
            return str(bio_card)
    return bio_card

def extract_confidence(prediction_json_str):
    """Extract confidence score from JSON response."""
    try:
        if pd.isna(prediction_json_str):
            return 'N/A'
        match = re.search(r'\{.*\}', str(prediction_json_str), re.DOTALL)
        if match:
            data = json.loads(match.group())
            return data.get('confidence', data.get('score', 'N/A'))
        return 'N/A'
    except Exception:
        return 'N/A'

def extract_emotion(prediction_json_str):
    """Extract emotion from JSON response."""
    try:
        if pd.isna(prediction_json_str):
            return 'unknown'
        match = re.search(r'\{.*\}', str(prediction_json_str), re.DOTALL)
        if match:
            data = json.loads(match.group())
            return data.get('predicted_emotion', 'unknown').lower().strip()
        return 'unknown'
    except Exception:
        return 'unknown'

speaker_bio_cards = load_speaker_bio_cards()
prediction_history = []
print(f"Available speakers: {list(speaker_bio_cards.keys())}")

# Create a unique key like "102_5" (Dialogue_ID + Utterance_ID + Season + Episode)
testing_data['Recognition_ID'] = (
    testing_data['Dialogue_ID'].astype(str) + '_' +
    testing_data['Utterance_ID'].astype(str) + '_' +
    testing_data['Season'].astype(str) + '_' +
    testing_data['Episode'].astype(str)
)

print(testing_data[['Dialogue_ID', 'Utterance_ID', 'Recognition_ID']].head())

def preprocess_test_data(testing_data):
    df = testing_data.copy()
    df = df.sort_values(['Dialogue_ID', 'Utterance_ID'])

    scenes = []
    for diag_id, group in df.groupby('Dialogue_ID'):
        agent_view = group[['Utterance', 'Speaker', 'Recognition_ID']].to_dict(orient='records')
        ground_truth = group[['Recognition_ID', 'Emotion', 'Sentiment']].to_dict(orient='records')

        scene_data = {
            'dialogue_id': diag_id,
            'utterances': agent_view,
            'ground_truth': ground_truth,
            'total_turns': len(group)
        }
        scenes.append(scene_data)
    return scenes

preprocessed_test_data = preprocess_test_data(testing_data)
preprocessed_test_data_df = pd.DataFrame(preprocessed_test_data)
print(preprocessed_test_data_df.head())
print(preprocessed_test_data[0])
print(f"Available keys in your data: {preprocessed_test_data[0]['utterances'][0].keys()}")

def run_phase3_council(scene_obj, global_prediction_history):
    """Processes a single scene using phase3 flow with Llama as final council aggregator."""
    dialogue_id = scene_obj['dialogue_id']
    utterances = scene_obj['utterances']
    ground_truth_map = {
        item['Recognition_ID']: {'Emotion': item['Emotion'], 'Sentiment': item['Sentiment']}
        for item in scene_obj.get('ground_truth', [])
    }

    scene_script = '\n'.join([f"{u['Speaker']}: {u['Utterance']}" for u in utterances])

    global_context = groq_llm_call(
        prompt=f"{context_manager_prompt}\n\nScene:\n{scene_script}",
        model='meta-llama/llama-4-scout-17b-16e-instruct',
        api_key=context_manager_api
    )
    social_graph = groq_llm_call(
        prompt=f"{relational_graph_prompt}\n\nScene:\n{scene_script}",
        model='meta-llama/llama-4-scout-17b-16e-instruct',
        api_key=relational_graph_api
    )

    def process_utterance(idx, u):
        try:
            target_text = u.get('Utterance', '')
            speaker = u.get('Speaker', 'Unknown')
            rec_id = u.get('Recognition_ID', 'unknown_id')
            actual_labels = ground_truth_map.get(rec_id, {'Emotion': 'neutral', 'Sentiment': 'neutral'})
            actual_emotion = actual_labels['Emotion']

            previous_turn = utterances[idx - 1] if idx > 0 else None
            previous_text = previous_turn.get('Utterance', '') if previous_turn else ''
            previous_speaker = previous_turn.get('Speaker', 'Unknown') if previous_turn else 'Unknown'

            with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
                f_profile = executor.submit(call_tuned_profiler, target_text, social_graph)
                f_sentiment = executor.submit(call_tuned_sentiment, target_text)
                f_shift = None

                if previous_turn is not None:
                    f_shift = executor.submit(
                        call_emotional_shift,
                        previous_text,
                        previous_speaker,
                        target_text,
                        speaker,
                        global_context
                    )

                profile_report = f_profile.result()
                sentiment_report = f_sentiment.result()

                if f_shift is not None:
                    emotional_shift_report = f_shift.result()
                else:
                    emotional_shift_report = '[SHIFT: FALSE - CONTINUITY MAINTAINED]'

                f_dynamics = executor.submit(
                    call_social_dynamics,
                    target_text,
                    profile_report,
                    social_graph
                )
                dynamics_report = f_dynamics.result()

            speaker_bio_card = get_speaker_bio_card_text(speaker, speaker_bio_cards)
            previous_predictions = global_prediction_history[-3:] if global_prediction_history else None

            final_prediction_raw = call_llama_council_aggregator(
                rec_id,
                target_text,
                global_context,
                profile_report,
                sentiment_report,
                dynamics_report,
                emotional_shift_report,
                speaker_bio_card=speaker_bio_card,
                previous_predictions=previous_predictions
            )

            return {
                'Dialogue_ID': dialogue_id,
                'Recognition_ID': rec_id,
                'Speaker': speaker,
                'Utterance': target_text,
                'Predicted_Emotion_Raw': final_prediction_raw,
                'Actual_Emotion': actual_emotion,
                'reasoning': profile_report,
                'predicted_emotion': extract_emotion(final_prediction_raw),
                'confidence': extract_confidence(final_prediction_raw),
                'emotional_shift_report': emotional_shift_report
            }
        except Exception as e:
            raise Exception(f"Error processing utterance {idx} (rec_id={u.get('Recognition_ID')}): {str(e)}")

    scene_results = []
    max_workers_utterance = min(5, len(utterances))

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers_utterance) as executor:
        futures = {executor.submit(process_utterance, idx, u): idx for idx, u in enumerate(utterances)}

        for future in concurrent.futures.as_completed(futures):
            result = future.result()
            scene_results.append(result)

            shift_flag = 'FALSE'
            try:
                shift_report = result.get('emotional_shift_report', '')
                if 'SHIFT: TRUE' in shift_report:
                    shift_flag = 'TRUE'
            except Exception:
                pass

            global_prediction_history.append({
                'utterance': result['Utterance'],
                'emotion': result.get('predicted_emotion', 'unknown'),
                'confidence': result.get('confidence', 'N/A'),
                'shift': shift_flag
            })
            if len(global_prediction_history) > 3:
                global_prediction_history.pop(0)

    scene_results.sort(
        key=lambda x: utterances[next(i for i, u in enumerate(utterances) if u['Recognition_ID'] == x['Recognition_ID'])]['Recognition_ID']
    )
    return scene_results

Loaded 260 speaker bio cards
Available speakers: ['1st Customer', '2nd Customer', '3rd Customer', 'A Female Student', 'A Student', 'Airline Employee', 'Alice', 'All', 'Allesandro', 'Angela', 'Annabelle', 'Another Scientist', 'Another Tour Guide', 'Aunt Lillian', 'Barry', 'Ben', 'Bernice', 'Bob', 'Bobby', 'Bonnie', 'Both', 'Boy in the Cape', 'Burt', 'Caitlin', 'Carl', 'Carol', 'Casey', 'Cassie', 'Cecilia', 'Chandler', 'Charlie', 'Charlton Heston', 'Chip', 'Chloe', 'Cliff', 'Commercial', 'Customer', 'Dana', 'Danny', 'David', 'Dina', 'Director', 'Doctor', 'Doug', 'Dr. Baldhara', 'Dr. Drake Remoray', 'Dr. Franzblau', 'Dr. Green', 'Dr. Johnson', 'Dr. Ledbetter', 'Dr. Leedbetter', 'Dr. Long', 'Dr. Miller', 'Dr. Oberman', 'Dr. Rhodes', 'Dr. Stryker Ramoray', 'Dr. Wesley', 'Dr. Zane', 'Drunken Gambler', 'Duncan', 'Earl', 'Elizabeth', 'Emeril', 'Emily', 'Eric', 'Estelle', 'Evil Bitch', 'Fake Monica', 'Fireman No. 1', 'Fireman No. 2', 'Fireman No. 3', 'Flight Attendant', 'Frank', 'Friend No. 1',

In [ ]:
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

output_file = '../logs/llama3/council_phase4_1_llama_results.csv'

processed_ids = set()
if os.path.exists(output_file):
    try:
        existing_df = pd.read_csv(output_file)
        if not existing_df.empty:
            processed_ids = set(existing_df['Dialogue_ID'].unique())
            print(f"Found {len(processed_ids)} scenes already processed. Skipping them...")
    except Exception as e:
        print(f"Could not read existing results: {e}")

all_final_results = []
stop_processing = threading.Event()
csv_lock = threading.Lock()
prediction_history = []

def process_scene_worker(scene, index, total, pred_history):
    """Worker function to process a single scene in parallel."""
    try:
        scene_id = scene['dialogue_id']
        print(f"[Worker] Processing Scene {index+1}/{total} (ID: {scene_id})...")
        scene_out = run_phase3_council(scene, pred_history)
        return (scene_id, scene_out, None)
    except Exception as e:
        error_msg = str(e).lower()
        if 'api key' in error_msg or 'unauthorized' in error_msg or '401' in error_msg:
            return (None, None, ('AUTH_ERROR', str(e)))
        elif '429' in error_msg or 'resource exhausted' in error_msg:
            return (None, None, ('RATE_LIMIT', str(e)))
        return (None, None, ('PROCESS_ERROR', str(e)))

scenes_to_process = [
    (i, scene) for i, scene in enumerate(preprocessed_test_data)
    if scene['dialogue_id'] not in processed_ids
]

print(f"Total unprocessed scenes: {len(scenes_to_process)}")

max_workers = 3
rate_limit_hit = False

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {
        executor.submit(process_scene_worker, scene, idx, len(scenes_to_process), prediction_history): (idx, scene)
        for idx, scene in scenes_to_process
    }

    completed = 0
    for future in as_completed(futures):
        if stop_processing.is_set():
            break

        idx, scene = futures[future]
        scene_id, scene_out, error = future.result()

        if error:
            error_type, error_msg = error
            if error_type == 'AUTH_ERROR':
                print(f"Authentication/config error at Scene {scene_id}. Fix credentials and rerun.")
                print(error_msg)
                stop_processing.set()
                break
            if error_type == 'RATE_LIMIT':
                print(f"Rate limit exhausted at Scene {scene_id}. Stopping gracefully.")
                print(error_msg)
                stop_processing.set()
                rate_limit_hit = True
                break
            print(f"Error in Scene {scene_id}: {error_msg}")
            continue

        if scene_out:
            with csv_lock:
                new_df = pd.DataFrame(scene_out)
                new_df.to_csv(output_file, mode='a', index=False, header=not os.path.exists(output_file))
                all_final_results.extend(scene_out)
                print(f"Scene {scene_id} saved to {output_file}")

        completed += 1
        print(f"Progress: {completed}/{len(scenes_to_process)} scenes completed")

if rate_limit_hit:
    print('\nPAUSED: Rate limit hit. Wait before re-running.')
else:
    print('\nSESSION FINISHED. All unprocessed scenes in this batch are done!')

Found 18 scenes already processed. Skipping them...
Total unprocessed scenes: 262
[Worker] Processing Scene 10/262 (ID: 9)...
[Worker] Processing Scene 12/262 (ID: 11)...
[Worker] Processing Scene 18/262 (ID: 17)...
[Worker] Processing Scene 22/262 (ID: 21)...
Scene 11 saved to ../logs/llama3/council_phase4_1_llama_results.csv
Progress: 1/262 scenes completed
[Worker] Processing Scene 23/262 (ID: 22)...
Scene 21 saved to ../logs/llama3/council_phase4_1_llama_results.csv
Progress: 2/262 scenes completed


In [1]:
import pandas as pd
import json
import re
from sklearn.metrics import classification_report, f1_score, confusion_matrix

def generate_erc_report(csv_path):
    df = pd.read_csv(csv_path)

    def extract_label(raw_string):
        """Extracts 'predicted_emotion' from the Aggregator's JSON string."""
        if pd.isna(raw_string):
            return 'neutral'
        try:
            clean_json = raw_string.replace('""', '"')
            match = re.search(r'\{.*\}', clean_json, re.DOTALL)
            if match:
                data = json.loads(match.group())
                return data.get('predicted_emotion', 'neutral').lower().strip()
            return 'neutral'
        except Exception:
            return 'neutral'

    df['Predicted_Clean'] = df['Predicted_Emotion_Raw'].apply(extract_label)
    df['Actual_Clean'] = df['Actual_Emotion'].str.lower().str.strip()

    labels = sorted(df['Actual_Clean'].unique())

    print('\n' + '=' * 60)
    print('      MULTI-AGENT COUNCIL: CLASSIFICATION REPORT')
    print('=' * 60)

    report = classification_report(
        df['Actual_Clean'],
        df['Predicted_Clean'],
        labels=labels,
        digits=4
    )
    print(report)

    wf1 = f1_score(df['Actual_Clean'], df['Predicted_Clean'], average='weighted')
    print(f"\nOVERALL WEIGHTED F1 SCORE: {wf1:.4f}")
    print('=' * 60)

    return df

report_df = generate_erc_report('../logs/llama3/council_phase4_1_llama_results.csv')


      MULTI-AGENT COUNCIL: CLASSIFICATION REPORT
              precision    recall  f1-score   support

       anger     0.5238    0.6111    0.5641        18
     disgust     0.5000    0.5000    0.5000         2
        fear     0.1818    0.6667    0.2857         3
         joy     0.3818    0.8077    0.5185        26
     neutral     0.7692    0.3797    0.5085        79
     sadness     0.1667    0.4286    0.2400         7
    surprise     1.0000    0.2143    0.3529        14

    accuracy                         0.4765       149
   macro avg     0.5033    0.5154    0.4243       149
weighted avg     0.6499    0.4765    0.4851       149


OVERALL WEIGHTED F1 SCORE: 0.4851
